In [10]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pyarrow.feather as feather
from scipy.stats import kstest

#import importlib
import sys
import os
hostname = os.uname().nodename
if hostname == 'BlackBeast':
    path = '/home/hedvigs/snake_book/econ'
    site = 'home'
elif hostname == 'hedvig-hp-elitedesk-800-g5-twr':
    path = '/home/hedvigs/PycharmProjects/homewrs/plab_workflow/'
    site = 'work'
elif hostname == 'work-computer':
    path = '/mnt/work/workbench/hedvigs/snake_book/econ'
    site = 'server'
elif hostname == 'SilverFlex':
    path = '/home/hedvigs/gitrepos/plab_workflow/'
    site = 'silverFlex'

sys.path.append(path)
print(path)
from workflow.scripts.data_management import setup_data as gt


/home/hedvigs/gitrepos/plab_workflow/


In [11]:
def summarize_scores(
    subset=None, model=None, gen=None, fold=None, path=None, verbose=0
):
    inputs = [subset, gen, fold, model]
    config_names = ["subsets", "genomes", "folds", "models"]

    config_dict = {}

    for item, config_name in zip(inputs, config_names):
        if item is not None:
            config_dict[config_name] = [item]
        else:
            config_dict[config_name] = gt.read_config(config_name, path=path)
    y_pred = []
    missing = pd.DataFrame()
    k = 0

    for subset in config_dict["subsets"]:
        for model in config_dict["models"]:
            for gen in config_dict["genomes"]:
                for fold in config_dict["folds"]:
                    if subset == "all":
                        score_file = (
                            path
                            + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}_pruned.csv"
                        )
                    else:
                        score_file = (
                            path
                            + f"results/scores/PTD/{subset}/{model}_{gen}_{fold}.csv"
                        )
                    try:
                        scores = pd.read_csv(score_file, sep="\t")
                        scores["fold"] = fold
                        scores["gen"] = gen
                        scores["model"] = model
                        scores["subset"] = subset
                        y_pred.append(scores)
                    except FileNotFoundError:
                        k += 1
                        missing.loc[k, "subset"] = subset
                        missing.loc[k, "model"] = model
                        missing.loc[k, "gen"] = gen
                        missing.loc[k, "fold"] = fold
    if k == 0:
        print(f" All files found!, {k} missing")
    elif verbose == 1:
        print(f"Files not found:", k)
    elif verbose == 2:
        print(f"Files not found:", k)
        subsetnames, subsetcounts = np.unique(missing["subset"], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f"  {sname}: {scount} missing")
            missubset = missing[missing["subset"] == sname]
            modelnames, modelcounts = np.unique(missubset["model"], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f"    {mname}: {mcount} missing")
    elif verbose == 3:
        print("Total files not found:", k)
        for colname in missing.columns:
            names, counts = np.unique(missing[colname], return_counts=True)
            print(f"{colname}: ")
            for name, count in zip(names, counts):
                print(f"  {name}: {count} missing")
    elif verbose == 4:
        print(f"Files not found:", k)
        subsetnames, subsetcounts = np.unique(missing["subset"], return_counts=True)
        for sname, scount in zip(subsetnames, subsetcounts):
            print(f"  {sname}: {scount} missing")
            missubset = missing[missing["subset"] == sname]
            modelnames, modelcounts = np.unique(missubset["model"], return_counts=True)
            for mname, mcount in zip(modelnames, modelcounts):
                print(f"    {mname}: {mcount} missing")
                missmodel = missubset[missubset["model"] == mname]
                gennames, gencounts = np.unique(missmodel["gen"], return_counts=True)
                for gname, gcount in zip(gennames, gencounts):
                    print(f"        {gname}: {gcount} missing")

    full_set = pd.concat(y_pred)
    return full_set

In [12]:
subset = None #'top29' # wildcards["iSubset"]
model = None
gen = None  # wildcards["iGen"]

sum_df = summarize_scores(subset=subset, model=model, gen=gen, fold=None,path=path, verbose=4)
auc_df = sum_df.drop(columns=["number"]) # ,"fold", "train_score", "f1", "fbeta", "perm_score", "plr", "nlr", "pval_score", "auc_pred"])


Files not found: 139
  all: 65 missing
    bnb: 2 missing
        m: 2 missing
    knn: 13 missing
        combine: 4 missing
        f: 4 missing
        m: 5 missing
    lda: 7 missing
        combine: 2 missing
        f: 1 missing
        m: 4 missing
    lrc: 11 missing
        combine: 3 missing
        f: 3 missing
        m: 5 missing
    nn: 10 missing
        combine: 5 missing
        f: 2 missing
        m: 3 missing
    qda: 6 missing
        combine: 3 missing
        m: 3 missing
    rfc: 7 missing
        combine: 4 missing
        f: 1 missing
        m: 2 missing
    svc: 9 missing
        combine: 4 missing
        f: 1 missing
        m: 4 missing
  top5: 74 missing
    bnb: 8 missing
        combine: 2 missing
        f: 2 missing
        m: 4 missing
    knn: 10 missing
        combine: 4 missing
        f: 2 missing
        m: 4 missing
    lda: 8 missing
        combine: 3 missing
        f: 3 missing
        m: 2 missing
    lrc: 8 missing
        combine: 2 mi

In [13]:

#sum_file= path + 'results/report/sum_file_top29.csv'
sum_file= path + f'results/report/sum_file_26.csv'
auc_df.to_csv(sum_file, sep="\t", index=False)
